In [ ]:
import logging

import itertools
from IPython.display import Markdown, display

from library.utils.faults.circuit_faults import analyse_heuristic
from library.circuitry import Circuitry, DetectorTag
from library.compact_magic_state_cultivation import MagicStateCultivation
from library.surface_code.patch import SurfaceCodePatch
import warnings

logging.basicConfig(level=logging.ERROR)
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
TARGET_DISTANCE = 7

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

TELEPORT_ROUNDS = 3
minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

In [ ]:
scenarios: dict[str, Circuitry] = dict()

## Circuit 0 : Injection

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=0, exp_rounds=0)

msc.steane.append_noiseless_measurement(circuitry)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Injected"] = circuitry

In [ ]:
scenarios["Injected"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 1 : Injection + Superdense

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=0)

msc.steane.annotate_stabilizers(circuitry, style="DIFFUSE", noiseless=True)
for stabilizer, color in itertools.product("XZ", "RGB"):
    circuitry.annotate_detector(
        f"STN:{stabilizer}{color}", tags=DetectorTag.POSTSELECTED
    )

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Superdense"] = circuitry

In [ ]:
# scenarios["Superdense"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 2 : Injection + Superdense + Double-check

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=0)

msc.steane.annotate_stabilizers(circuitry, style="DIFFUSE", noiseless=True)
for stabilizer, color in itertools.product("XZ", "RGB"):
    circuitry.annotate_detector(
        f"STN:{stabilizer}{color}", tags=DetectorTag.POSTSELECTED
    )

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Double-check"] = circuitry

In [ ]:
# scenarios["Double-checked"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 3 : Injection + Superdense + Double-check + Teleportation

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)
msc.append_teleportation(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=0)

msc.source.annotate_stabilizers(circuitry, prefix="SRC:STB", noiseless=True)
circuitry.annotate_detector(
    "JCT:TPT2:X0", "SRC:STB:X0", tags=DetectorTag.COMPLEMENTARY_GAP
)
for stabilizer in "XZ":
    for location, (_, index) in msc.source.qubits[stabilizer].items():
        tags = DetectorTag.COMPLEMENTARY_GAP
        if msc.critical_on_seam(stabilizer, location):
            tags |= DetectorTag.POSTSELECTED
        circuitry.annotate_detector(
            f"SRC:TPT2:{stabilizer}{index}", f"SRC:STB:{stabilizer}{index}", tags=tags
        )

msc.source.append_decode_measurement(
    circuitry,
    prefix="SRC:DCD",
    noiseless=True,
    corrections=(f"JCT:TPT2:{mz}" for mz in ["Z0", "Z1", "Z2"]),
)

circuitry.attach_observable(
    0,
    "SRC:DCD:X",
    *(f"STN:TPT2:X{mx}" for mx in [1, 5, 6]),
)

scenarios["Teleported"] = circuitry

In [ ]:
# scenarios["Teleported"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 4 : Injection + Superdense + Double-check + Teleportation + Expansion

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)
msc.append_teleportation(circuitry)
exp_rounds = msc.append_expansion(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=exp_rounds)

msc.target.annotate_stabilizers(circuitry, prefix="TGT:STB", noiseless=True)
for stabilizer, index in itertools.product(
    "XZ", range((msc.target.distance**2 - 1) // 2)
):
    circuitry.annotate_detector(
        f"TGT:EXP:R{exp_rounds - 1}:{stabilizer}{index}",
        f"TGT:STB:{stabilizer}{index}",
        tags=DetectorTag.COMPLEMENTARY_GAP,
    )

z_byproducts, x_byproducts = msc.expansion_byproducts()
msc.target.append_decode_measurement(
    circuitry,
    prefix="TGT:DCD",
    noiseless=True,
    corrections=[*(f"JCT:TPT2:Z{i}" for i in range(3)), *z_byproducts],
)

circuitry.attach_observable(
    0, "TGT:DCD:X", *(f"STN:TPT2:X{mx}" for mx in [1, 5, 6]), *x_byproducts
)

scenarios["Expanded"] = circuitry

In [ ]:
# scenarios["Expanded"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Summary

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    print(f"Point {index} - {scenario}")
    circuitry.detectors_report()

### Circuits

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    label = f"Point {index} - {scenario}"
    display(Markdown(f"[Open in Crumble ({label})]({circuitry.to_crumble_url()})"))

### Fault analysis (S-proxies)

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    label = f"Point {index} - {scenario}"
    analyse_heuristic(circuitry.as_stim(per=0.01), label)

## Simulation and sampling [S-proxies]

In [ ]:
from library.utils.plotting import plot_ler_curve
from library.utils.stim.simulation import compute_ler_curve as stim_compute_ler
from library.utils.stim.sampling import sample as stim_sample
from library.utils.clifft.stratified_mc_reweighted import (
    compute_ler_curves as clifft_compute_ler,
)
from library.utils.clifft.sampling import sample as clifft_sample
from library.utils.clifft.plotting import (
    plot_logical_error_rates as clifft_plot_logical_error_rates,
)
import numpy

In [ ]:
FILEROOT = "generated/s-proxies/hirano-magic-state-cultivation-layout1-compact"
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    circuitry.to_file(FILEROOT + f".point{index}.{scenario.lower()}")

In [ ]:
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [$\overline{\mathbf{Y}}$-observable, noiseless]"

In [ ]:
stim_sample(
    scenarios, title=title, label="Point", shots=1e6, correction=True, fontsize=10
)

In [ ]:
result_files = stim_compute_ler(
    scenarios,
    label="Point",
    postselection=True,
    shots=1e7,
    minimal_noise=-7,
    num_workers=7,
    exclude=["Teleported", "Expanded"],
)

In [ ]:
plot_ler_curve(result_files, title, figsize=(12, 4.5), estimate_distance=True)

In [ ]:
noise_levels = numpy.logspace(-7, -2, num=11)
results_ler, results_cer = clifft_compute_ler(
    scenarios,
    label="Point",
    shots_per_stratum=2e7,
    batch_count=400,
    max_errors=100,
    noise_levels=noise_levels,
    gap_threshold=48.0,
)

In [ ]:
clifft_plot_logical_error_rates(
    results_ler, results_cer, "Point", estimate_distance=True
)

## Simulation and sampling [T-gates]

In [ ]:
title = r"Magic State Cultivation of $|\mathbf{T}\rangle$ [$\frac{\overline{\mathbf{X}}+\overline{\mathbf{Z}}}{\sqrt{2}}$-observable, noiseless]"

In [ ]:
FILEROOT = "generated/t-gates/hirano-magic-state-cultivation-layout1-compact"
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    circuitry.to_file(
        FILEROOT + f".point{index}.{scenario.lower()}",
        proxy_replacements=MagicStateCultivation.NON_CLIFFORD_PROXY_REPLACEMENTS,
    )

In [ ]:
# Sample the outcomes of the circuits at various points
clifft_sample(
    scenarios,
    proxy_replacements=MagicStateCultivation.NON_CLIFFORD_PROXY_REPLACEMENTS,
    title=title,
    label="Point",
    shots=1e6,
    correction=True,
    postselection=True,
    figsize=(16, 5),
    fontsize=8,
)

In [ ]:
noise_levels = numpy.logspace(-7, -2, num=11)
results_ler, results_cer = clifft_compute_ler(
    scenarios,
    label="Point",
    shots_per_stratum=2e7,
    batch_count=400,
    max_errors=100,
    noise_levels=noise_levels,
    gap_threshold=48.0,
    proxy_replacements=MagicStateCultivation.NON_CLIFFORD_PROXY_REPLACEMENTS,
)

In [ ]:
clifft_plot_logical_error_rates(
    results_ler, results_cer, "Point", noise_levels, estimate_distance=True
)